# 🎬 ViniVideo AI — Kaggle Qualidade (Wan 2.1)

Este notebook é o modo **Qualidade** do ViniVideo AI.

Ele usa **Wan 2.1 T2V 1.3B** com dois perfis: **Qualidade Rápida** e **Máxima**. Na v0.6, o **prompt do usuário tem prioridade** e o notebook só acrescenta instruções técnicas de estilo e consistência.

## Antes de rodar

1. Kaggle → **Settings → Accelerator → GPU T4 x2**.
2. Deixe **Internet** ativada.
3. Use **Run All**.
4. Comece com **5 segundos**. Depois teste 10 segundos.

> Importante: este modelo não é o Gemini Omni 1.1 Flash. Ele é uma alternativa aberta para priorizar qualidade dentro da GPU gratuita do Kaggle.


### Fluxo v0.6
No app, toque em **Gerar no Kaggle Qualidade**. O app copia um bloco de configuração; cole esse bloco na célula **CONFIGURAÇÃO** antes de executar a geração.


In [ ]:
import os
import torch

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "YES"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Ative Settings → Accelerator → GPU T4 x2 no Kaggle."
    )

print("GPU 0:", torch.cuda.get_device_name(0))
print("Quantidade de GPUs:", torch.cuda.device_count())


In [ ]:
!pip -q install -U "diffusers>=0.35,<0.41" "transformers>=4.48,<4.58" "accelerate>=1.2,<2" "huggingface-hub>=0.30" safetensors ftfy imageio imageio-ffmpeg hf-xet sentencepiece
print("Dependências de qualidade prontas ✓")


In [ ]:
import gc
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from transformers import UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

print("🎬 Carregando Wan 2.1 1.3B...")
print("O primeiro carregamento é mais demorado que o modo Turbo.")

text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    vae=vae,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=3.0,
)

pipe.enable_model_cpu_offload(gpu_id=0)

print("✅ Wan 2.1 pronto")


In [ ]:
# ===== CONFIGURAÇÃO =====
# Cole aqui o bloco copiado pelo app ViniVideo AI v0.6.
# O seu prompt é a prioridade; o notebook só adiciona reforços técnicos.
USER_PROMPT = """Uma arara azul conhece um novo amigo em uma praça tropical."""
STYLE = "cartoon_movie"   # cartoon_movie, default, anime, cinematic, realistic_3d
ASPECT = "9:16"           # "9:16" ou "16:9"
DURATION_SECONDS = 5
PROFILE = "RAPIDO_QUALIDADE"   # RAPIDO_QUALIDADE ou MAXIMA
SEED = 20261007

STYLE_PRESETS = {
    "cartoon_movie": (
        "stylized 3D animated feature-film look, clean appealing character shapes, "
        "expressive cartoon facial animation, soft stylized materials, colorful production design, "
        "cinematic but clearly animated lighting, smooth coherent character motion, "
        "avoid uncanny photorealism"
    ),
    "default": (
        "clean high-quality animation, stable character identity, coherent motion"
    ),
    "anime": (
        "anime-inspired animation, clean stylized character design, expressive motion, "
        "consistent proportions and colors"
    ),
    "cinematic": (
        "cinematic animated film look, controlled lighting, appealing composition, "
        "smooth camera movement and coherent motion"
    ),
    "realistic_3d": (
        "high-quality stylized 3D rendering, cinematic lighting, coherent anatomy, "
        "stable character identity"
    ),
}

def build_prompt(user_prompt, style):
    base = user_prompt.strip()
    style_text = STYLE_PRESETS.get(style, STYLE_PRESETS["default"])
    return (
        base
        + ". "
        + style_text
        + ". Preserve the same character identity, face, proportions, colors and clothing "
          "throughout the entire shot. Follow the user's requested action and setting closely. "
          "Smooth coherent animation, natural camera motion, no random cuts."
    )

PROMPT = build_prompt(USER_PROMPT, STYLE)

NEGATIVE_PROMPT = """low quality, worst quality, blurry, flicker, jitter,
warped face, deformed face, changing eyes, changing character design,
duplicated body parts, extra limbs, extra wings, deformed beak, broken anatomy,
melting anatomy, inconsistent proportions, random cuts, camera teleport,
uncanny photorealism, photorealistic skin, text, watermark, noisy image,
frozen motion, compression artifacts"""

if PROFILE == "MAXIMA":
    STEPS = 28
    NUM_FRAMES = 81
    VERTICAL_SIZE = (448, 800)
    HORIZONTAL_SIZE = (800, 448)
else:
    STEPS = 18
    NUM_FRAMES = 65
    VERTICAL_SIZE = (384, 672)
    HORIZONTAL_SIZE = (672, 384)

GUIDANCE = 4.5
SMOOTH_24FPS = False

print("Prompt do usuário:", USER_PROMPT)
print("Estilo:", STYLE)
print("Perfil:", PROFILE)
print("Duração:", DURATION_SECONDS, "s")
print("Formato:", ASPECT)
print("Steps:", STEPS, "| Frames:", NUM_FRAMES)


In [ ]:
from pathlib import Path
from diffusers.utils import export_to_video
from IPython.display import Video, display, FileLink
import subprocess
import torch
import gc
import math
import time

OUTPUT_DIR = Path("/kaggle/working/ViniVideoAI_Quality")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def dimensions(aspect):
    if aspect == "9:16":
        return VERTICAL_SIZE
    return HORIZONTAL_SIZE

width, height = dimensions(ASPECT)

scene_seconds = 5
scene_count = max(1, math.ceil(int(DURATION_SECONDS) / scene_seconds))
scene_paths = []

gpu_count = torch.cuda.device_count()
print()
print(f"🎞️ Gerando {scene_count} tomada(s) em {width}x{height}...")
print(f"Perfil: {PROFILE} • {STEPS} steps • {NUM_FRAMES} frames")
print("GPUs disponíveis:", gpu_count)

if gpu_count >= 2 and scene_count >= 2:
    print("ℹ️ T4 x2 detectada. Nesta versão, a segunda T4 fica disponível para futuros lotes paralelos.")
    print("O gargalo principal de uma tomada única é a difusão sequencial do Wan.")

total_started = time.time()

for scene_index in range(scene_count):
    scene_started = time.time()
    start = scene_index * scene_seconds
    remaining = int(DURATION_SECONDS) - start
    this_duration = max(1, min(scene_seconds, remaining))

    scene_path = OUTPUT_DIR / f"wan_scene_{scene_index + 1:03d}.mp4"

    if scene_path.exists() and scene_path.stat().st_size > 2048:
        print(f"♻️ Tomada {scene_index + 1} já existe; reutilizando.")
        scene_paths.append(scene_path)
        continue

    continuity = (
        " Continue the exact same visual identity, lighting, palette, "
        "character proportions and camera language from the previous shot."
        if scene_index > 0 else ""
    )

    scene_prompt = (
        PROMPT
        + continuity
        + f" This is shot {scene_index + 1} of {scene_count}."
    )

    generator = torch.Generator(device="cpu").manual_seed(
        int(SEED) + scene_index
    )

    print(
        f"🎬 Tomada {scene_index + 1}/{scene_count} "
        f"• {this_duration}s • {NUM_FRAMES} frames • {STEPS} steps"
    )

    def progress(pipe, step, timestep, kwargs):
        pct = int(((step + 1) / max(1, STEPS)) * 100)
        if step == 0 or step + 1 == STEPS or pct % 10 == 0:
            elapsed = int(time.time() - scene_started)
            print(f"   Difusão: {pct}% • {elapsed}s")
        return kwargs

    result = pipe(
        prompt=scene_prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=height,
        width=width,
        num_frames=int(NUM_FRAMES),
        num_inference_steps=int(STEPS),
        guidance_scale=float(GUIDANCE),
        generator=generator,
        callback_on_step_end=progress,
    )

    frames = result.frames[0]

    native_fps = max(
        8,
        min(
            16,
            round(len(frames) / max(1, this_duration))
        )
    )

    export_to_video(
        frames,
        str(scene_path),
        fps=native_fps,
    )

    scene_paths.append(scene_path)

    del result, frames
    gc.collect()
    torch.cuda.empty_cache()

    elapsed = int(time.time() - scene_started)
    print(f"✅ Tomada {scene_index + 1} salva em {elapsed}s")

concat_file = OUTPUT_DIR / "concat.txt"
with open(concat_file, "w", encoding="utf-8") as f:
    for path in scene_paths:
        safe = str(path).replace("'", "'\\''")
        f.write(f"file '{safe}'\n")

raw_path = OUTPUT_DIR / "ViniVideoAI_Wan_raw.mp4"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(concat_file),
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        str(raw_path),
    ],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

final_path = OUTPUT_DIR / "ViniVideoAI_Wan_24fps.mp4"

if ASPECT == "9:16":
    resize = (
        "scale=360:640:force_original_aspect_ratio=increase,"
        "crop=360:640"
    )
else:
    resize = (
        "scale=640:360:force_original_aspect_ratio=increase,"
        "crop=640:360"
    )

if SMOOTH_24FPS:
    fps_filter = (
        "minterpolate=fps=24:mi_mode=mci:mc_mode=aobmc:"
        "me_mode=bidir:vsbmc=1,"
    )
    print("🎞️ Fazendo interpolação suave para 24 fps...")
else:
    fps_filter = "fps=24,"
    print("⚡ Convertendo para 24 fps no modo rápido...")

vf = fps_filter + resize

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(raw_path),
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(final_path),
    ],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

total_elapsed = int(time.time() - total_started)

print()
print("✅ VÍDEO QUALIDADE PRONTO")
print("Tempo total:", total_elapsed, "s")
print("Arquivo:", final_path)

display(Video(str(final_path), embed=True))
display(
    FileLink(
        str(final_path),
        result_html_prefix="⬇️ Baixar MP4 qualidade: "
    )
)


## O que este modo melhora

- Modelo de vídeo real **Wan 2.1 1.3B**, em vez de AnimateDiff.
- Perfil **Qualidade Rápida**: 384×672 (9:16), 65 frames e 18 steps.
- Perfil **Máxima**: 448×800, 81 frames e 28 steps.
- Saída final em **24 fps**; interpolação pesada fica opcional.
- Checkpoint por tomada: se a célula final for executada de novo na mesma sessão, tomadas já concluídas podem ser reutilizadas.

### Limitação atual

O Wan 1.3B deste notebook é **texto→vídeo**. Ele melhora bastante a qualidade visual, mas ainda não tem o mesmo controle multimodal e áudio nativo do Gemini Omni 1.1 Flash.

A próxima etapa para consistência de personagem por imagem é integrar **Wan VACE 1.3B** ou um pipeline de referência.


### v0.6 — prompt fiel e estilo

- `USER_PROMPT` é usado como núcleo da geração.
- `STYLE = "cartoon_movie"` reforça visual de filme animado e evita semi-realismo estranho.
- O negative prompt combate deformação de rosto, olhos, bico, anatomia e flicker.
- O MP4 pode ser baixado no Kaggle e depois importado no projeto pelo app.
